In [7]:
import re
from lxml import etree
import html
import os # Import the os module

# --- Configuration ---
# Directory containing your XML files
XML_DIR = '../GRC_misc/'
# Directory where you want to save the HTML output
OUTPUT_DIR = './html_output/'
# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# List of play identifiers (used to build filenames)
PLAY_IDS = ['seven', 'pb', 'supp', 'pers', 'ag', 'lib', 'eum']

# Build the full input file paths
XML_FILES = [os.path.join(XML_DIR, f'aesch.{pid}.wecklein1885-grc2.xml') for pid in PLAY_IDS]

# XML namespaces
ns = {'tei': 'http://www.tei-c.org/ns/1.0'}

print(f"Found {len(XML_FILES)} XML files to process.")
print(f"HTML output will be saved in '{OUTPUT_DIR}'.")

Found 7 XML files to process.
HTML output will be saved in './html_output/'.


In [8]:
def process_play(xml_filepath, html_output_filepath):
    """
    Parses an Aeschylus TEI XML file, extracts text/app crit/scholia,
    and generates a three-column HTML file.
    """
    print(f"\nProcessing '{os.path.basename(xml_filepath)}'...")
    try:
        # --- 1. Parsing ---
        parser = etree.XMLParser(remove_blank_text=True)
        root = etree.parse(xml_filepath, parser)
        print("   - XML parsed.")

        # --- 2. Data Extraction ---
        greek_col = {}
        app_crit_col = {}
        commentary_col = {} # Scholia
        lemmas_to_highlight = {}

        # Pre-process Lemmas
        for app in root.findall('.//tei:app[@loc]', ns):
            line_num_str = app.get('loc')
            if line_num_str and line_num_str.isdigit():
                lem = app.find('tei:lem', ns)
                if lem is not None:
                    lem_text = "".join(lem.itertext()).strip()
                    if lem_text:
                        lemmas_to_highlight.setdefault(line_num_str, []).append(lem_text)

        # Extract Greek Text
        for line in root.findall('.//tei:l[@n]', ns):
            line_num_str = line.get('n')
            if line_num_str:
                line_text_parts = [line.text or '']
                for child in line:
                     line_text_parts.append(etree.tostring(child, encoding='unicode', method='text'))
                line_text = ''.join(line_text_parts).strip()
                line_text_cleaned = re.sub('<[^>]+>', '', line_text)
                highlighted_text = html.escape(line_text_cleaned)
                if line_num_str in lemmas_to_highlight:
                    for lem in lemmas_to_highlight[line_num_str]:
                        escaped_lem = html.escape(lem)
                        highlighted_text = re.sub(r'\b' + re.escape(escaped_lem) + r'\b',
                                                 f'<span class="highlight-lem">{escaped_lem}</span>', highlighted_text)
                speaker_tag = line.find('../tei:speaker', ns)
                speaker_html = ''
                if speaker_tag is not None and speaker_tag.text is not None:
                    prev_sibling = line.getprevious()
                    is_new_speech = prev_sibling is None or prev_sibling.tag != '{http://www.tei-c.org/ns/1.0}l'
                    if is_new_speech:
                        speaker_html = f'<span class="speaker">{html.escape(speaker_tag.text)}</span> '
                html_content = f'<div class="line" id="g-{line_num_str}"><a class="line-num" data-line="{line_num_str}">{line_num_str}</a> <span class="text">{speaker_html}{highlighted_text}</span></div>'
                greek_col[line_num_str] = html_content

        # Extract App Crit
        for app in root.findall('.//tei:app[@loc]', ns):
            line_num_str = app.get('loc')
            if line_num_str and line_num_str.isdigit():
                content_parts = []
                lem = app.find('tei:lem', ns)
                if lem is not None: content_parts.append(f'<span class="lemma">{html.escape("".join(lem.itertext()))}</span>:')
                for rdg in app.findall('tei:rdg', ns):
                    rdg_text = "".join(rdg.itertext()).strip()
                    wit = rdg.get('wit', '')
                    content_parts.append(f'{html.escape(rdg_text)} <span class="witness">[{wit}]</span>' if wit else html.escape(rdg_text))
                app_text = ' '.join(content_parts).strip()
                html_content = f'<div class="line" id="app-{line_num_str}"><a class="line-num" data-line="{line_num_str}">{line_num_str}</a> <span class="text">{app_text}</span></div>'
                app_crit_col.setdefault(line_num_str, []).append(html_content)

        # Extract Scholia
        for note in root.findall('.//tei:div[@type="scholia"]//tei:note[@target]', ns):
            target_attr = note.get('target')
            if target_attr:
                line_nums_target = re.findall(r'#l\.(\d+)', target_attr)
                note_text = ''.join(note.itertext()).strip()
                if not note_text: continue
                note_text_display = html.escape(note_text)
                line_ref_html = ''
                note_line_ref_match = re.match(r'^\s*([\d\.\s–-]+)\s*', note_text)
                if note_line_ref_match:
                    line_ref_num = note_line_ref_match.group(1).strip()
                    if line_ref_num.endswith('.') or line_ref_num.replace('-', '').isdigit():
                        line_ref_html = f'<span class="line-ref">{html.escape(line_ref_num)}</span> '
                        note_text_display = html.escape(note_text[note_line_ref_match.end():].strip())
                for line_num_target in line_nums_target:
                    html_content = f'<div class="line" id="com-{line_num_target}"><a class="line-num" data-line="{line_num_target}">{line_num_target}</a> <span class="text">{line_ref_html}{note_text_display}</span></div>'
                    commentary_col.setdefault(line_num_target, []).append(html_content)

        print(f"   - Data extracted: {len(greek_col)} lines, {sum(len(v) for v in app_crit_col.values())} app crit, {sum(len(v) for v in commentary_col.values())} scholia.")

        # --- 3. HTML Generation ---
        max_line_num = 0
        all_keys = list(greek_col.keys()) + list(app_crit_col.keys()) + list(commentary_col.keys())
        if all_keys:
            valid_keys = [int(k) for k in all_keys if k.isdigit()]
            if valid_keys: max_line_num = max(valid_keys)

        greek_html = "\n".join([greek_col.get(str(i), f'<div class="line empty"><a class="line-num" data-line="{i}">{i}</a></div>') for i in range(1, max_line_num + 1)])
        app_crit_lines = []
        for i in range(1, max_line_num + 1):
            if str(i) in app_crit_col: app_crit_lines.extend(app_crit_col[str(i)])
        app_crit_html = "\n".join(app_crit_lines)
        commentary_lines_builder = []
        for i in range(1, max_line_num + 1):
             if str(i) in commentary_col: commentary_lines_builder.extend(commentary_col[str(i)])
        commentary_html = "\n".join(commentary_lines_builder)

        title_tag = root.find('.//tei:titleStmt/tei:title', ns)
        main_title = title_tag.text if title_tag is not None else "Untitled Document"
        author_tag = root.find('.//tei:titleStmt/tei:author', ns)
        author = author_tag.text if author_tag is not None else "Unknown Author"
        
        # --- NEW: Extract Editor and Date ---
        editor_tag = root.find('.//tei:titleStmt/tei:editor', ns)
        editor = editor_tag.text if editor_tag is not None else "Unknown Editor"
        date_tag = root.find('.//tei:publicationStmt/tei:date', ns)
        pub_date = date_tag.text if date_tag is not None else "Unknown Date"
        # --- END NEW ---

        # (Your HTML template string - kept concise here for brevity)
        html_template = f"""
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-R-8">
            <meta name="viewport" content="width=device-width, initial-scale=1.0">
            <title>{main_title}</title>
            <style>
                /* Your CSS styles here */
                body {{ font-family: 'Georgia', serif; display: flex; flex-direction: column; height: 100vh; margin: 0; background-color: #fdfdfd; }}
                header {{ padding: 10px 20px; border-bottom: 2px solid #ddd; background-color: #fff; text-align: center; }}
                h1 {{ margin: 0; font-size: 1.8em; color: #333; }}
                h2 {{ margin: 5px 0 0; font-size: 1.2em; color: #666; font-style: italic; font-weight: normal;}}
                .container {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; flex-grow: 1; padding: 10px; overflow: hidden; }}
                .column {{ background-color: #ffffff; border: 1px solid #e0e0e0; border-radius: 4px; display: flex; flex-direction: column; overflow: hidden; }}
                .column h3 {{ text-align: center; margin: 0; padding: 12px; border-bottom: 1px solid #e0e0e0; background-color: #f9f9f9; color: #444; font-size: 1em; }}
                .content {{ padding: 10px; overflow-y: auto; height: 100%; }}
                .line {{ display: flex; align-items: baseline; padding: 3px 5px; border-radius: 3px; min-height: 1.5em; }}
                #app-crit-content .line, #commentary-content .line {{ min-height: auto; }}
                #greek-content .line.empty .text {{ color: #ccc; }}
                .line.highlight {{ background-color: #e7f5ff; }}
                .line-num {{ flex-shrink: 0; width: 40px; font-size: 0.8em; color: #888; cursor: pointer; text-align: right; margin-right: 10px; font-family: monospace; }}
                .line-num:hover {{ color: #007bff; }}
                .text {{ line-height: 1.6; }}
                .speaker {{ font-weight: bold; margin-right: 8px; color: #800000; }}
                .lemma {{ font-weight: bold; font-style: italic; color: #0056b3; }}
                .witness {{ font-style: italic; color: #555; font-size: 0.9em; }}
                .line-ref {{ font-weight: bold; color: #555; margin-right: 4px; }}
                .highlight-lem {{ background-color: #fff9c4; border-radius: 3px; padding: 0 2px; }}
            </style>
        </head>
        <body>
            <header>
                <h1>{main_title}</h1>
                <h2>{author} | Editor: {editor} ({pub_date})</h2>
            </header>
            <div class="container">
                <div class="column"><h3>Greek Text</h3><div class="content" id="greek-content">{greek_html}</div></div>
                <div class="column"><h3>Apparatus Criticus</h3><div class="content" id="app-crit-content">{app_crit_html}</div></div>
                <div class="column"><h3>Scholia</h3><div class="content" id="commentary-content">{commentary_html}</div></div>
            </div>
            <script>
                document.addEventListener('DOMContentLoaded', function() {{
                    const container = document.querySelector('.container');
                    let lastClickedLine = null;
                    container.addEventListener('click', function(event) {{
                        if (event.target.classList.contains('line-num')) {{
                            const lineNumber = event.target.dataset.line;
                            if(lastClickedLine) {{ document.querySelectorAll(`.line.highlight`).forEach(el => el.classList.remove('highlight')); }}
                            const prefixes = ['g', 'app', 'com'];
                            prefixes.forEach(prefix => {{
                                const element = document.getElementById(`${{prefix}}-${{lineNumber}}`);
                                if (element) {{ element.scrollIntoView({{ behavior: 'smooth', block: 'center' }}); element.classList.add('highlight'); }}
                            }});
                            lastClickedLine = lineNumber;
                        }}
                    }});
                }});
            </script>
        </body>
        </html>
        """

        # --- 4. Write to file ---
        with open(html_output_filepath, 'w', encoding='utf-8') as f:
            f.write(html_template)
        print(f"   - Successfully created '{os.path.basename(html_output_filepath)}'.")

    except IOError:
        print(f"   - Error: File not found '{xml_filepath}'.")
    except etree.XMLSyntaxError as e:
        print(f"   - Error: XML syntax error in '{xml_filepath}'. {e}")
    except Exception as e:
        print(f"   - Error: An unexpected error occurred processing '{xml_filepath}'. {e}")

In [9]:
# Loop through all the defined XML files
for i, xml_file in enumerate(XML_FILES):
    play_id = PLAY_IDS[i] # Get the short identifier (e.g., 'seven', 'pb')
    # Construct the output filename
    output_html = os.path.join(OUTPUT_DIR, f'aeschylus_{play_id}.html')
    # Process the current play
    process_play(xml_file, output_html)

print("\nProcessing complete for all plays.")


Processing 'aesch.seven.wecklein1885-grc2.xml'...
   - XML parsed.
   - Data extracted: 1070 lines, 500 app crit, 575 scholia.
   - Successfully created 'aeschylus_seven.html'.

Processing 'aesch.pb.wecklein1885-grc2.xml'...
   - XML parsed.
   - Data extracted: 1128 lines, 421 app crit, 487 scholia.
   - Successfully created 'aeschylus_pb.html'.

Processing 'aesch.supp.wecklein1885-grc2.xml'...
   - XML parsed.
   - Data extracted: 1085 lines, 519 app crit, 378 scholia.
   - Successfully created 'aeschylus_supp.html'.

Processing 'aesch.pers.wecklein1885-grc2.xml'...
   - XML parsed.
   - Data extracted: 1077 lines, 212 app crit, 227 scholia.
   - Successfully created 'aeschylus_pers.html'.

Processing 'aesch.ag.wecklein1885-grc2.xml'...
   - XML parsed.
   - Data extracted: 1673 lines, 704 app crit, 315 scholia.
   - Successfully created 'aeschylus_ag.html'.

Processing 'aesch.lib.wecklein1885-grc2.xml'...
   - XML parsed.
   - Data extracted: 1078 lines, 613 app crit, 529 scholia.
